# Stock Price Prediction Using Support Vector Machines
**Undergraduate Final Year Project | Matrusri Engineering College (2023)**

**Published in:** International Journal of Creative Research Thoughts (IJCRT)  
**ISSN:** 2320-2882 | **Impact Factor:** 7.97 | **UGC Approved**  
**Paper ID:** IJCRT2305597 | **Volume 11, Issue 5 | May 2023**

---

## 📌 Project Overview
This project investigates the effectiveness of **Support Vector Machines (SVM)** 
for predicting stock price movements on NSE Reliance Industries data. The SVM 
model is benchmarked against traditional approaches (ANN, LSTM) and evaluated 
on its practical viability for everyday stock prediction.

**Key Question:** Can SVM's linear and non-linear problem-solving capabilities 
produce meaningful predictions for volatile stock market data?

---

## 🔑 Key Results
- **SVM Accuracy: 56%** — outperforming the 50% random baseline
- SVM demonstrated competitive performance vs. ANN and LSTM with significantly 
  lower computational cost
- Feature engineering (Open-Close, High-Low differentials) proved effective 
  as predictive signals for binary movement classification

---

## 📁 Table of Contents
1. [Imports & Setup](#1-imports--setup)
2. [Load & Explore Dataset](#2-load--explore-dataset)
3. [Feature Engineering](#3-feature-engineering)
4. [Train/Test Split](#4-traintest-split)
5. [Model Training — SVM](#5-model-training--svm)
6. [Strategy Returns Calculation](#6-strategy-returns-calculation)
7. [Visualization — Cumulative Returns](#7-visualization--cumulative-returns)
8. [Model Evaluation](#8-model-evaluation)
9. [Conclusions](#9-conclusions)


## 1. Imports & Setup
Import all required libraries for machine learning, data manipulation, and visualization.


In [ ]:
# Machine Learning
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-darkgrid')

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")


## 2. Load & Explore Dataset

**Dataset:** NSE Reliance Industries historical stock data  
**Source:** Yahoo Finance  
**Features:** Date, Open, High, Low, Close, Volume

The dataset is loaded and the Date column is set as the index for time-series analysis.


In [ ]:
# Load the dataset
df = pd.read_csv('RELIANCE_NS.csv')

# Set Date as index
df.index = pd.to_datetime(df['Date'])
df = df.drop(['Date'], axis='columns')

# Preview the data
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print("\nFirst 5 rows:")
df.head()


In [ ]:
# Basic statistical overview of the dataset
print("=== Dataset Info ===")
df.info()
print("\n=== Statistical Summary ===")
df.describe()


In [ ]:
# Check for missing values
print("=== Missing Values per Column ===")
print(df.isnull().sum())


In [ ]:
# Visualize closing price over time
plt.figure(figsize=(14, 5))
plt.plot(df['Close'], color='steelblue', linewidth=1.2)
plt.title('Reliance Industries — Closing Price Over Time', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Closing Price (INR)')
plt.tight_layout()
plt.show()


## 3. Feature Engineering

Two engineered features are created as predictive signals:
- **Open-Close:** Difference between opening and closing price — captures intraday momentum
- **High-Low:** Difference between daily high and low — captures intraday volatility

**Target variable:** Binary classification — 1 if next day's closing price is higher, 0 otherwise.


In [ ]:
# Create predictor features
df['Open-Close'] = df['Open'] - df['Close']
df['High-Low'] = df['High'] - df['Low']

# Store predictor variables
X = df[['Open-Close', 'High-Low']]

# Target variable: 1 if next day's close > today's close, else 0
y = np.where(df['Close'].shift(-1) > df['Close'], 1, 0)

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution — Up: {sum(y==1)} | Down: {sum(y==0)}")
print("\nFeature preview:")
X.head(10)


In [ ]:
# Visualize the engineered features
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df.index, df['Open-Close'], color='coral', linewidth=0.8, alpha=0.8)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=0.5)
axes[0].set_title('Open-Close Differential (Intraday Momentum)')
axes[0].set_ylabel('Price Difference (INR)')

axes[1].plot(df.index, df['High-Low'], color='steelblue', linewidth=0.8, alpha=0.8)
axes[1].set_title('High-Low Differential (Intraday Volatility)')
axes[1].set_ylabel('Price Difference (INR)')

plt.tight_layout()
plt.show()


## 4. Train/Test Split

The dataset is split **chronologically** (not randomly) using an 80/20 ratio.  
Chronological splitting is critical for time-series data to prevent data leakage 
— the model must never train on future data.

- **Training set:** First 80% of records
- **Test set:** Last 20% of records


In [ ]:
# Chronological train/test split (67/33)
split_percentage = 0.67
split = int(split_percentage * len(df))

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

print(f"Total records:     {len(df)}")
print(f"Training records:  {len(X_train)} ({split_percentage*100:.0f}%)")
print(f"Testing records:   {len(X_test)} ({(1-split_percentage)*100:.0f}%)")
print(f"\nTraining period: {df.index[0].date()} to {df.index[split-1].date()}")
print(f"Testing period:  {df.index[split].date()} to {df.index[-1].date()}")


## 5. Model Training — Support Vector Machine (SVM)

The **Support Vector Classifier (SVC)** from Scikit-learn is used with default 
parameters (RBF kernel). SVM works by finding the optimal hyperplane that 
maximizes the margin between the two classes (price up vs. price down).

**Why SVM for stock prediction?**
- Effective in high-dimensional spaces
- Robust to overfitting in small-to-medium datasets
- Handles non-linear decision boundaries via kernel trick (RBF kernel)
- Lower computational cost vs. ANN/LSTM


In [ ]:
# Train the SVM classifier
cls = SVC(kernel='rbf', random_state=42)
cls.fit(X_train, y_train)

# Generate predictions on the full dataset for strategy backtesting
df['Predicted_Signal'] = cls.predict(X)

# Generate predictions on test set for evaluation
y_pred = cls.predict(X_test)

print("SVM model trained successfully.")
print(f"Kernel: {cls.kernel}")
print(f"Support vectors per class: {cls.n_support_}")


## 6. Strategy Returns Calculation

The SVM predictions are used to simulate a trading strategy:
- **Signal = 1 (Up):** Buy — capture the day's return
- **Signal = 0 (Down):** Stay out — avoid the day's loss

Returns are calculated and compared against a simple buy-and-hold strategy.


In [ ]:
# Calculate daily returns
df['Return'] = df['Close'].pct_change()

# Calculate strategy returns based on SVM predicted signals
df['Strategy_Return'] = df['Return'] * df['Predicted_Signal'].shift(1)

# Calculate cumulative returns for both strategies
df['Cum_Ret'] = df['Return'].cumsum()
df['Cum_Strategy'] = df['Strategy_Return'].cumsum()

print("Returns calculated successfully.")
print(f"\nTotal buy-and-hold return: {df['Cum_Ret'].iloc[-1]*100:.2f}%")
print(f"Total SVM strategy return: {df['Cum_Strategy'].iloc[-1]*100:.2f}%")


## 7. Visualization — Cumulative Returns

Comparing the SVM trading strategy against a simple buy-and-hold approach 
over the full historical period.


In [ ]:
# Plot cumulative returns comparison
plt.figure(figsize=(14, 6))
plt.plot(df['Cum_Ret'], color='red', linewidth=1.2, label='Buy & Hold Strategy')
plt.plot(df['Cum_Strategy'], color='blue', linewidth=1.2, label='SVM Strategy')
plt.title('Cumulative Returns: SVM Strategy vs. Buy & Hold\nReliance Industries (NSE)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize predicted vs actual signals on a sample period
sample = df.iloc[split:split+100].copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(sample.index, sample['Close'], color='steelblue', linewidth=1.2)
ax1.set_title('Closing Price — Test Period (Sample: 100 days)')
ax1.set_ylabel('Price (INR)')

ax2.plot(sample.index, y_test[:100], color='green', 
         linewidth=1, alpha=0.7, label='Actual Signal')
ax2.plot(sample.index, cls.predict(X_test[:100]), color='red', 
         linewidth=1, alpha=0.7, linestyle='--', label='Predicted Signal')
ax2.set_title('Actual vs. Predicted Signals (1=Up, 0=Down)')
ax2.set_ylabel('Signal')
ax2.legend()

plt.tight_layout()
plt.show()


## 8. Model Evaluation

Evaluating the SVM model performance on the held-out test set using 
standard classification metrics.


In [ ]:
# Model accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"=== SVM Model Performance ===")
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Random Baseline: 50.00%")
print(f"Improvement over baseline: +{(accuracy-0.5)*100:.2f}%")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, 
      target_names=['Price Down (0)', 'Price Up (1)']))


In [ ]:
# Confusion matrix visualization
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Down', 'Predicted Up'],
            yticklabels=['Actual Down', 'Actual Up'])
plt.title('Confusion Matrix — SVM Stock Prediction', fontsize=13)
plt.tight_layout()
plt.show()

print(f"True Positives (Correctly predicted Up):   {cm[1][1]}")
print(f"True Negatives (Correctly predicted Down): {cm[0][0]}")
print(f"False Positives (Predicted Up, was Down):  {cm[0][1]}")
print(f"False Negatives (Predicted Down, was Up):  {cm[1][0]}")


## 9. Conclusions

### Key Findings
1. **SVM achieved 56% accuracy** on binary stock movement classification — 
   outperforming the 50% random baseline
2. **Feature engineering** (Open-Close and High-Low differentials) provided 
   meaningful predictive signal for intraday momentum and volatility
3. **SVM vs. ANN/LSTM:** SVM demonstrated competitive predictive performance 
   with significantly lower computational requirements — making it practical 
   for real-time applications with limited resources
4. The **RBF kernel** effectively handled the non-linear relationships in 
   stock price data

### Limitations & Future Work
- A 56% accuracy, while above baseline, may not be sufficient for real-world 
  trading without risk management
- **Future improvements:**
  - Add more technical indicators (RSI, MACD, Bollinger Bands)
  - Experiment with different SVM kernels (polynomial, sigmoid)
  - Incorporate volume and sentiment data as additional features
  - Compare with ensemble methods (Random Forest, Gradient Boosting)
  - Apply to multiple stocks for generalizability testing

### Publication
This research was peer-reviewed and published in the **International Journal 
of Creative Research Thoughts (IJCRT)**, Impact Factor 7.97, May 2023.  
Paper ID: IJCRT2305597

---

*Project by Batch 5 — Matrusri Engineering College, Hyderabad, India (2023)*
